# 附录 D：训练循环增强（Bells & Whistles）

> ch05 写了最朴素的训练循环。本附录加上生产级训练的标配：**学习率调度、训练/验证划分、早停、梯度裁剪、checkpoint**。

> 这些不是花架子——真实训练 LLM 必备，缺一不可。

## 1. 学习率调度器（最重要）

固定学习率效果差。现代 LLM 标配：**warmup（线性升温）+ cosine decay（余弦衰减）**。

- **warmup**：初期 lr 从小到大，避免随机初始化时 loss 爆炸
- **cosine decay**：后期 lr 余弦降到接近 0，让训练稳定收敛

In [ ]:
import math
import torch


class CosineWithWarmup:
    """warmup 线性升 → cosine 余弦降到 min_lr。"""

    def __init__(self, optimizer, num_warmup, num_training, base_lr, min_lr_ratio=0.1):
        self.optimizer = optimizer
        self.num_warmup = num_warmup
        self.num_training = num_training
        self.base_lr = base_lr
        self.min_lr = base_lr * min_lr_ratio
        self.step_num = 0

    def step(self):
        self.step_num += 1
        if self.step_num < self.num_warmup:
            lr = self.base_lr * self.step_num / self.num_warmup
        else:
            progress = (self.step_num - self.num_warmup) / (self.num_training - self.num_warmup)
            lr = self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))
        for group in self.optimizer.param_groups:
            group["lr"] = lr
        return lr


# 可视化 lr 曲线
base_lr, warmup, total = 3e-4, 50, 500
params = [torch.randn(3, requires_grad=True)]
opt = torch.optim.AdamW(params, lr=base_lr)
sched = CosineWithWarmup(opt, warmup, total, base_lr)
lrs = [sched.step() for _ in range(total)]

print(f"base_lr={base_lr}, warmup={warmup}, total={total}")
print(f"{'step':<8} {'lr':<12}")
for i in range(0, total, 50):
    bar = "█" * int(lrs[i] / base_lr * 30)
    print(f"{i:<8} {lrs[i]:.6f}  {bar}")
print("\n💡 先升后降：warmup 防初期发散，cosine 让后期精细收敛。")

## 2. 训练/验证划分 + 早停

真实训练要划分 train/val，监控 val loss。**早停**：val loss 连续 N 轮不降就停，防过拟合。

In [ ]:
def train_with_early_stopping(model, train_loader, val_loader, optimizer,
                              num_epochs, patience=5):
    """带早停的训练循环。"""
    best_val_loss = float("inf")
    epochs_no_improve = 0
    
    for epoch in range(num_epochs):
        # 训练
        model.train()
        train_loss = 0; n = 0
        for x, y in train_loader:
            optimizer.zero_grad()
            loss = torch.nn.functional.mse_loss(model(x).squeeze(), y.float())
            loss.backward(); optimizer.step()
            train_loss += loss.item(); n += 1
        train_loss /= n
        
        # 验证
        model.eval()
        val_loss = 0; m = 0
        with torch.no_grad():
            for x, y in val_loader:
                val_loss += torch.nn.functional.mse_loss(
                    model(x).squeeze(), y.float()).item(); m += 1
        val_loss /= m
        
        print(f"Epoch {epoch+1}: train {train_loss:.4f} | val {val_loss:.4f}")
        
        # 早停逻辑
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"早停：val loss 连续 {patience} 轮未改善，停止训练。")
                break
    return best_val_loss


# demo：小数据 + 简单模型
from torch.utils.data import TensorDataset, DataLoader
torch.manual_seed(0)
X = torch.randn(100, 4); Y = (X.sum(1) > 0).long()
train_ds = TensorDataset(X[:80], Y[:80]); val_ds = TensorDataset(X[80:], Y[80:])
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=16)
model = torch.nn.Linear(4, 1)
opt = torch.optim.AdamW(model.parameters(), lr=0.01)
train_with_early_stopping(model, train_dl, val_dl, opt, num_epochs=20, patience=3)

## 3. 梯度裁剪（防梯度爆炸）

深层网络（如 GPT）训练时梯度可能爆炸，导致 loss 变 NaN。**梯度裁剪**：限制梯度的范数上限，稳定训练。

In [ ]:
import torch.nn as nn

model = nn.Linear(10, 1)
x = torch.randn(4, 10) * 100      # 放大输入制造大梯度
y = torch.randn(4)
opt = torch.optim.AdamW(model.parameters(), lr=0.1)

opt.zero_grad()
loss = nn.functional.mse_loss(model(x).squeeze(), y)
loss.backward()

# 裁剪前的梯度范数
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f"梯度范数（裁剪前）: {grad_norm.item():.2f}")
print(f"已裁剪到 max_norm=1.0（防梯度爆炸）")
opt.step()
print("\n💡 LLM 训练标配：max_norm 通常设 1.0，防止 loss 突然变 NaN。")

## 4. Checkpoint 保存与恢复

长训练可能中断。定期保存 checkpoint（模型+优化器+epoch），中断后能从断点恢复。

In [ ]:
import os

def save_checkpoint(model, optimizer, epoch, path):
    """保存训练状态。"""
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
    }, path)
    print(f"已保存 checkpoint (epoch {epoch}) → {path}")

def load_checkpoint(model, optimizer, path):
    """恢复训练状态。"""
    ckpt = torch.load(path, weights_only=True)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    print(f"已从 {path} 恢复 (epoch {ckpt['epoch']})")
    return ckpt["epoch"]

# demo
model = nn.Linear(10, 1)
opt = torch.optim.AdamW(model.parameters(), lr=0.01)
save_checkpoint(model, opt, epoch=10, path="data/ckpt_demo.pt")
load_checkpoint(model, opt, path="data/ckpt_demo.pt")
os.remove("data/ckpt_demo.pt")
print("\n💡 真实训练每隔 N 步保存一次，崩溃后 load 最新 checkpoint 继续。")

---
> **小结**：lr 调度器（warmup+cosine）+ train/val 划分 + 早停 + 梯度裁剪 + checkpoint。
> 这五件套把 ch05 的朴素循环升级成生产级训练流水线。